# 7. Auditoria Comparativa de Erros e Análise de Discrepância Multi-Modelo

Esta secção implementa uma pipeline avançada de diagnóstico de Engenharia de Machine Learning (*Error Analysis*). Em vez de avaliar o modelo de forma holística através de métricas macro (como mAP global), este algoritmo realiza uma auditoria microscópica e comparativa entre sete arquiteturas distintas treinadas ao longo do projeto (`yolov8` e `yolov11` em várias escalas).

O objetivo principal é isolar sistematicamente todas as instâncias do subconjunto de teste onde ocorrem falhas de generalização, categorizando-as em:
1. **Falsos Negativos:** Ocorrências reais (*Ground Truth*) negligenciadas pelo modelo (`sem_detecao`).
2. **Falsos Positivos / Erros de Classificação:** Discrepâncias em que a classe prevista diverge da anotação estipulada no vetor de etiquetas.

O algoritmo executa uma validação formal (`model.val()`) para gerar matrizes de predição em formato COCO (`predictions.json`), agrupa as inferências por identificador de imagem através de hashing em memória, e realiza uma comparação booliana de conjuntos ordenados contra os ficheiros descritores nativos `.txt`. As imagens que violam a correspondência exata são renderizadas via OpenCV e segregadas em diretorias dinâmicas baseadas na assinatura do erro, permitindo um diagnóstico visual imediato das fragilidades de cada rede.

In [ ]:
import os
import json
import cv2
import shutil
from pathlib import Path
from ultralytics import YOLO

# ── 1. Configuração de Infraestrutura e Mapeamento de Checkpoints ──────────────
DATA_YAML   = "/content/Trabalho_IA-2/data.yaml"
TEST_IMAGES = Path("/content/Trabalho_IA-2/test/images")
TEST_LABELS = Path("/content/Trabalho_IA-2/test/labels")
OUTPUT_DIR  = Path("/content/errors")

DRIVE_BASE = "drive/MyDrive/Colab Notebooks/gloves_project/runs"

# Dicionário estruturado mapeando o identificador do modelo ao seu respetivo arquivo de pesos (.pt)
MODELS = {
    "yolov8n_tatica_a" : f"{DRIVE_BASE}/yolov8n_tatica_a/weights/best.pt",
    "yolov8n_tatica_b2": f"{DRIVE_BASE}/yolov8n_tatica_b2/weights/best.pt",
    "yolov8n_tatica_c" : f"{DRIVE_BASE}/yolov8n_tatica_c/weights/best.pt",
    "yolov8s_fase2"    : f"{DRIVE_BASE}/yolov8s_fase2/weights/best.pt",
    "yolov8m_fase2"    : f"{DRIVE_BASE}/yolov8m_fase2/weights/best.pt",
    "yolov11s_fase2"   : f"{DRIVE_BASE}/yolov11s_fase2/weights/best.pt",
    "yoloV11m_fase2"   : f"{DRIVE_BASE}/yoloV11m_fase2/weights/best.pt",
}

# ── 2. Loop de Avaliação Sistemática da População de Modelos ──────────────────
for name, weights_path in MODELS.items():
    print(f"\n{'='*60}")
    print(f"A processar análise de erro para a arquitetura: {name}")
    print(f"{'='*60}")

    # Instanciar a rede com a topologia e pesos específicos da iteração
    model = YOLO(weights_path)

     # Executar a rotina de validação focada no split de teste para gerar o mapa de predições COCO    
    metrics = model.val(
        data=DATA_YAML,
        split="test",
        imgsz=640,
        conf=0.001,      # Limiar ultra-baixo para capturar todas as ativações potenciais da rede
        iou=0.6,         # Limiar de interseção sobre união para supressão de não-máximos (NMS)
        plots=False,     # Desativar gráficos padrão para otimizar velocidade de processamento
        save_json=True,  # Parâmetro crítico: força a exportação do ficheiro 'predictions.json'
        project="/content/val_results",
        name=name,
        verbose=False,
    )

    # Validar a existência física do ficheiro de predições serializado    
    pred_json = Path(f"/content/val_results/{name}/predictions.json")
    if not pred_json.exists():
        print(f"[AVISO] Ficheiro predictions.json não localizado para o modelo {name}")
        continue

    with open(pred_json) as f:
        predictions = json.load(f)

    # ── 3. Agrupamento e Filtragem Estatística de Predições por Amostra ───────    preds_by_image = {}
    for pred in predictions:
        img_id = pred["image_id"]
        if img_id not in preds_by_image:
            preds_by_image[img_id] = []
        
        # Filtro de confiança operacional: ignora ruídos matemáticos abaixo de 30%
        if pred["score"] >= 0.3:  
            # O formato COCO incrementa +1 ao ID da classe em relação ao formato nativo do YOLO
            preds_by_image[img_id].append(pred["category_id"] - 1)

    # Criar a diretoria de isolamento de falhas para o modelo corrente    model_output = OUTPUT_DIR / name
    model_output.mkdir(parents=True, exist_ok=True)

    # ── 4. Análise de Discrepância Espacial e de Classe (Amostra por Amostra) ──
    for img_path in sorted(TEST_IMAGES.iterdir()):
        if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue

        label_path = TEST_LABELS / (img_path.stem + ".txt")
        if not label_path.exists():
            continue

        # Extração do Ground Truth (Verdade de Campo) a partir do ficheiro de anotações original        gt_classes = []
        for line in label_path.read_text().splitlines():
            parts = line.strip().split()
            if parts:
                gt_classes.append(int(parts[0]))

        # Capturar as predições validadas pelo modelo para a chave da imagem correspondente
        img_id = img_path.stem
        pred_classes = preds_by_image.get(img_id, [])

        # ── 5. Comparação Lógica de Vetores de Classe ─────────────────────────
        # Ordenamos os vetores para garantir que a comparação valida o conteúdo e não a sequência de deteção        
        if sorted(gt_classes) != sorted(pred_classes):
            # Mapeamento reverso dos IDs numéricos para strings textuais legíveis pelo utilizador
            gt_names   = [model.names[c] for c in sorted(set(gt_classes))]
            pred_names = [model.names[c] for c in sorted(set(pred_classes))] if pred_classes else ["sem_detecao"]

            # Geração da assinatura de erro taxonómica para nomenclatura da pasta
            folder_name = f"gt_{'_'.join(gt_names)}__pred_{'_'.join(pred_names)}"
            folder = model_output / folder_name
            folder.mkdir(exist_ok=True)

            # Executar uma inferência visual isolada para gerar a imagem anotada com caixas
            result = model.predict(
                source=str(img_path),
                conf=0.001,
                iou=0.6,
                imgsz=640,
                verbose=False,
            )[0]

            # Gerar a matriz BGR sobreposta com as falhas detetadas pelo modelo
            annotated = result.plot(conf=True, labels=True, boxes=True)

            # Persistir a evidência gráfica do erro no respetivo cluster de falha
            cv2.imwrite(str(folder / img_path.name), annotated)

    print(f"-> Auditoria concluída. Artefactos de falha guardados em: {model_output}")

# ── 6. Consolidação e Extração Automatizada de Dados de Auditoria ─────────────
print("\n[Auditoria] A compactar o repositório de erros para transferência...")
shutil.make_archive("/content/errors", "zip", "/content/errors")

# Invocar a API de I/O do Colab para descarregar o ficheiro .zip para a máquina local
from google.colab import files
files.download("/content/errors.zip")